In [3]:
import duckdb
import pandas as pd

# Configuration Pandas pour afficher de longs textes (les descriptions de jobs)
pd.set_option('display.max_colwidth', 100)
pd.set_option('display.max_columns', None)

# Connexion à la base (le chemin remonte d'un dossier depuis /notebooks)
con = duckdb.connect('../jobradar.duckdb')

print("✅ Connecté à DuckDB avec succès !")

✅ Connecté à DuckDB avec succès !


In [4]:
# La commande PRAGMA est le standard DuckDB pour lister les tables
display(con.execute("PRAGMA show_tables").df())

,name
0,raw_adzuna
1,raw_france_travail


In [5]:
# On navigue dans le schéma JSON d'Adzuna
query_adzuna = """
    SELECT 
        job_data.id AS id_offre,
        job_data.title AS titre,
        job_data.company.display_name AS entreprise,
        job_data.location.display_name AS lieu,
        job_data.salary_min AS salaire_min,
        job_data.description AS description
    FROM raw_adzuna
    LIMIT 5
"""
print("--- PREVIEW ADZUNA ---")
display(con.execute(query_adzuna).df())

--- PREVIEW ADZUNA ---


,id_offre,titre,entreprise,lieu,salaire_min,description
0,5698744892,Apprenti(e) Data Analyst Marketing F/H,BPM GROUP,"Orvault, Nantes",<NA>,BPM Group est un acteur majeur de la distribution automobile et des services de mobilité. Dans u...
1,5619427068,Data Analyst (F/H),GROUPE DERET,"Nantes, Loire-Atlantique",<NA>,Offre d’emploi – Data Analyst / BI Qlik (H/F) Nantes CDI À propos de DERET Rattaché(e) à l’ent...
2,5681137168,Data Analyst F/H,CARREVOLUTIS,"Nantes, Loire-Atlantique",<NA>,"Description de l'entreprise Nous recrutons pour l'un de nos clients, société de services informa..."
3,5698743327,Alternance - Data Analyst (H/F) Alternance / Apprentissage H/F,Bloom Alternance,"Nantes, Loire-Atlantique",<NA>,Data builder & analyste métier * Créer et maintenir des tableaux de bord pour la finance et les ...
4,5676614310,Data Analyste Senior - H/F,Talan,"Nantes, Loire-Atlantique",<NA>,Description du poste Au sein de l'équipe Data Intelligence de Talan Nantes déja composés de 70 e...


In [12]:
# On navigue dans le schéma JSON de France Travail (API v2)
query_ft = """
    SELECT 
        job_data.id AS id_offre,
        job_data.intitule AS titre,
        job_data.entreprise.nom AS entreprise,
        job_data.lieuTravail.libelle AS lieu,
        job_data.salaire.libelle AS infos_salaire,
        job_data.description AS description
    FROM raw_france_travail
    LIMIT 5
"""
print("--- PREVIEW FRANCE TRAVAIL ---")
display(con.execute(query_ft).df())

--- PREVIEW FRANCE TRAVAIL ---


,id_offre,titre,entreprise,lieu,infos_salaire,description
0,206XRBG,Data analyst (F/H),EXPECTRA - APPEL MEDICAL,44 - Nantes,"""Annuel de 26500.0 Euros sur 12.0 mois""",Quel défi passionnant vous incite à embrasser le rôle de Master Data Administrateur (F/H) ?\nAu ...
1,1514955,Data Analyst en alternance F/H (H/F),Caisse d'Epargne Bretagne Pays de Loire,44 - Orvault,None,Poste et missions\n\nNous sommes à la recherche de notre futur.e Data Analyst en alternance pour...
2,1472886,Data Analyst Expérimenté (F/H),None,44 - Loire-Atlantique,None,"\n Le poste en quelques mots : Dans notre entité Technology Services, nous pro..."
3,1455708,Jean Rouyer Automobiles - Data analyst (H/F),None,44 - Nantes,None,"Ça vous dirait de mettre le turbo dans votre carrière ? C'est le bon moment, devenez Data Analys..."
4,1428525,Data Analyst Expérimenté (F/H),None,44 - Loire-Atlantique,None,"\n Le poste en quelques mots : Dans notre entité Technology Services, nous pro..."


In [13]:
query_quality = """
    SELECT 
        'Adzuna' AS source,
        COUNT(*) AS total,
        COUNT(job_data.salary_min) AS avec_salaire,
        ROUND(COUNT(job_data.salary_min) * 100.0 / COUNT(*), 1) AS pourcentage_avec_salaire
    FROM raw_adzuna
    
    UNION ALL
    
    SELECT 
        'France Travail' AS source,
        COUNT(*) AS total,
        COUNT(job_data.salaire.libelle) AS avec_salaire,
        ROUND(COUNT(job_data.salaire.libelle) * 100.0 / COUNT(*), 1) AS pourcentage_avec_salaire
    FROM raw_france_travail
"""
print("--- ANALYSE DE QUALITÉ (SALAIRES) ---")
display(con.execute(query_quality).df())

--- ANALYSE DE QUALITÉ (SALAIRES) ---


,source,total,avec_salaire,pourcentage_avec_salaire
0,Adzuna,237,107,45.1
1,France Travail,340,64,18.8
